In [3]:
from pathlib import Path
import duckdb
import pandas as pd

# =========================================================
# PATHS
# =========================================================

# =========================================================
# PROJECT ROOT
# =========================================================

import os
from pathlib import Path


def resolve_project_root():

    # Used automatically by run_pipeline.py
    env_root = os.environ.get(
        "DATA_ENGINEERING_R62_ROOT"
    )

    if env_root:
        root = Path(env_root).resolve()

        if (
            (root / "data").exists()
            and
            (root / "notebooks").exists()
        ):
            return root


    # Interactive notebook execution
    cwd = Path.cwd().resolve()

    candidates = [
        cwd,
        cwd.parent
    ]

    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):
            return candidate


    raise RuntimeError(
        "Could not locate DATA_ENGINEERING_R62 project root."
    )


PROJECT_ROOT = resolve_project_root()

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "\n✅ SUCCESS — Project root resolved."
)

PROCESSED = PROJECT_ROOT / "data" / "processed"
FEATURES = PROJECT_ROOT / "data" / "features"

USER_DAY = PROCESSED / "user_day_r62_v1.parquet"
BASELINE = FEATURES / "user_baseline_r62_v1.parquet"

BEHAVIOR_FILE = FEATURES / "behavior_features_r62_v1.parquet"

user_day_path = str(USER_DAY).replace("\\", "/")
baseline_path = str(BASELINE).replace("\\", "/")
behavior_path = str(BEHAVIOR_FILE).replace("\\", "/")

CUTOFF = "2011-01-01"

assert USER_DAY.exists()
assert BASELINE.exists()

print("User-Day :", USER_DAY)
print("Baseline :", BASELINE)
print("Output   :", BEHAVIOR_FILE)
print("Cutoff   :", CUTOFF)

print("\n✅ SUCCESS — Behavior-feature setup ready.")

Project root: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62

✅ SUCCESS — Project root resolved.
User-Day : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\user_day_r62_v1.parquet
Baseline : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\features\user_baseline_r62_v1.parquet
Output   : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\features\behavior_features_r62_v1.parquet
Cutoff   : 2011-01-01

✅ SUCCESS — Behavior-feature setup ready.


In [5]:
# =========================================================
# BUILD BASELINE-RELATIVE FEATURES
# =========================================================

con = duckdb.connect()

con.execute(f"""
COPY (

WITH future_days AS (

    SELECT
        d.*,

        -- =================================================
        -- BASELINE COVERAGE
        -- =================================================

        b.baseline_workdays,

        b.email_active_days,
        b.email_active_rate,

        b.email_count_p50,
        b.email_count_p90,
        b.email_count_p95,
        b.email_count_mad,

        b.email_bytes_p50,
        b.email_bytes_p90,
        b.email_bytes_p95,
        b.email_bytes_mad,

        b.usb_active_days,
        b.usb_active_rate,

        b.usb_copy_p50,
        b.usb_copy_p90,
        b.usb_copy_p95,
        b.usb_copy_mad,

        b.http_upload_active_days,
        b.http_upload_active_rate,

        b.http_upload_p50,
        b.http_upload_p90,
        b.http_upload_p95,
        b.http_upload_mad,


        -- =================================================
        -- CHANNEL ACTIVE TODAY?
        -- =================================================

        CASE
            WHEN d.external_email_count > 0
            THEN 1 ELSE 0
        END AS email_active_today,

        CASE
            WHEN d.usb_copy_count > 0
            THEN 1 ELSE 0
        END AS usb_active_today,

        CASE
            WHEN d.www_upload_count > 0
            THEN 1 ELSE 0
        END AS http_upload_active_today,


        -- =================================================
        -- NUMBER OF ACTIVE OUTBOUND CHANNELS TODAY
        -- =================================================

        (
            CASE
                WHEN d.external_email_count > 0
                THEN 1 ELSE 0
            END

            +

            CASE
                WHEN d.usb_copy_count > 0
                THEN 1 ELSE 0
            END

            +

            CASE
                WHEN d.www_upload_count > 0
                THEN 1 ELSE 0
            END
        ) AS active_channel_count,


        -- =================================================
        -- ABOVE USER-SPECIFIC P95?
        -- =================================================

        CASE
            WHEN b.email_count_p95 IS NOT NULL
             AND d.external_email_count > b.email_count_p95
            THEN 1 ELSE 0
        END AS email_count_above_p95,

        CASE
            WHEN b.email_bytes_p95 IS NOT NULL
             AND d.external_email_bytes > b.email_bytes_p95
            THEN 1 ELSE 0
        END AS email_bytes_above_p95,

        CASE
            WHEN b.usb_copy_p95 IS NOT NULL
             AND d.usb_copy_count > b.usb_copy_p95
            THEN 1 ELSE 0
        END AS usb_copy_above_p95,

        CASE
            WHEN b.http_upload_p95 IS NOT NULL
             AND d.www_upload_count > b.http_upload_p95
            THEN 1 ELSE 0
        END AS http_upload_above_p95,


        -- =================================================
        -- RATIO TO USER P95
        -- =================================================

        CASE
            WHEN b.email_count_p95 > 0
            THEN
                d.external_email_count::DOUBLE
                / b.email_count_p95
            ELSE NULL
        END AS email_count_to_p95_ratio,

        CASE
            WHEN b.email_bytes_p95 > 0
            THEN
                d.external_email_bytes::DOUBLE
                / b.email_bytes_p95
            ELSE NULL
        END AS email_bytes_to_p95_ratio,

        CASE
            WHEN b.usb_copy_p95 > 0
            THEN
                d.usb_copy_count::DOUBLE
                / b.usb_copy_p95
            ELSE NULL
        END AS usb_copy_to_p95_ratio,

        CASE
            WHEN b.http_upload_p95 > 0
            THEN
                d.www_upload_count::DOUBLE
                / b.http_upload_p95
            ELSE NULL
        END AS http_upload_to_p95_ratio


    FROM read_parquet(
        '{user_day_path}'
    ) d

    INNER JOIN read_parquet(
        '{baseline_path}'
    ) b
        ON d.user = b.user

    WHERE d.day >= DATE '{CUTOFF}'
      AND d.is_weekend = 0
      AND d.logon_count > 0
),


-- =========================================================
-- CAUSAL ROLLING FEATURES
--
-- wd = WORKDAYS
--
-- 5wd  = current workday + previous 4 workdays
-- 10wd = current workday + previous 9 workdays
-- 20wd = current workday + previous 19 workdays
--
-- No future rows are used.
-- =========================================================

rolling AS (

    SELECT
        *,

        -- =================================================
        -- EMAIL ACTIVE DAYS
        -- =================================================

        SUM(email_active_today) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 4 PRECEDING
                     AND CURRENT ROW
        ) AS email_active_days_5wd,

        SUM(email_active_today) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 9 PRECEDING
                     AND CURRENT ROW
        ) AS email_active_days_10wd,

        SUM(email_active_today) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 19 PRECEDING
                     AND CURRENT ROW
        ) AS email_active_days_20wd,


        -- =================================================
        -- USB ACTIVE DAYS
        -- =================================================

        SUM(usb_active_today) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 4 PRECEDING
                     AND CURRENT ROW
        ) AS usb_active_days_5wd,

        SUM(usb_active_today) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 9 PRECEDING
                     AND CURRENT ROW
        ) AS usb_active_days_10wd,

        SUM(usb_active_today) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 19 PRECEDING
                     AND CURRENT ROW
        ) AS usb_active_days_20wd,


        -- =================================================
        -- HTTP UPLOAD ACTIVE DAYS
        -- =================================================

        SUM(http_upload_active_today) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 4 PRECEDING
                     AND CURRENT ROW
        ) AS http_active_days_5wd,

        SUM(http_upload_active_today) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 9 PRECEDING
                     AND CURRENT ROW
        ) AS http_active_days_10wd,

        SUM(http_upload_active_today) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 19 PRECEDING
                     AND CURRENT ROW
        ) AS http_active_days_20wd,


        -- =================================================
        -- TRANSFER COUNTS - 5 WORKDAYS
        -- =================================================

        SUM(external_email_count) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 4 PRECEDING
                     AND CURRENT ROW
        ) AS external_email_count_5wd,

        SUM(usb_copy_count) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 4 PRECEDING
                     AND CURRENT ROW
        ) AS usb_copy_count_5wd,

        SUM(www_upload_count) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 4 PRECEDING
                     AND CURRENT ROW
        ) AS http_upload_count_5wd,


        -- =================================================
        -- TRANSFER COUNTS - 10 WORKDAYS
        -- =================================================

        SUM(external_email_count) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 9 PRECEDING
                     AND CURRENT ROW
        ) AS external_email_count_10wd,

        SUM(usb_copy_count) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 9 PRECEDING
                     AND CURRENT ROW
        ) AS usb_copy_count_10wd,

        SUM(www_upload_count) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 9 PRECEDING
                     AND CURRENT ROW
        ) AS http_upload_count_10wd,


        -- =================================================
        -- CROSS-CHANNEL ACTIVITY
        -- =================================================

        SUM(
            CASE
                WHEN active_channel_count >= 2
                THEN 1
                ELSE 0
            END
        ) OVER (
            PARTITION BY user
            ORDER BY day
            ROWS BETWEEN 9 PRECEDING
                     AND CURRENT ROW
        ) AS multi_channel_days_10wd

    FROM future_days
)


SELECT *

FROM rolling

ORDER BY
    user,
    day

)

TO '{behavior_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

con.close()

assert BEHAVIOR_FILE.exists()

print("Created:")
print(BEHAVIOR_FILE)

print(
    "\n✅ SUCCESS — behavior_features_r62_v1.parquet created."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\features\behavior_features_r62_v1.parquet

✅ SUCCESS — behavior_features_r62_v1.parquet created.


In [7]:
# =========================================================
# BEHAVIOR FEATURE VALIDATION
# =========================================================

con = duckdb.connect()

summary = con.execute(f"""
SELECT

    COUNT(*) AS rows,

    COUNT(DISTINCT user) AS users,

    MIN(day) AS first_day,

    MAX(day) AS last_day,

    MIN(baseline_workdays)
        AS min_baseline_workdays,

    MEDIAN(baseline_workdays)
        AS median_baseline_workdays,

    MAX(baseline_workdays)
        AS max_baseline_workdays

FROM read_parquet(
    '{behavior_path}'
)
""").df()


duplicates = con.execute(f"""
SELECT COUNT(*)

FROM (

    SELECT
        user,
        day,
        COUNT(*) AS n

    FROM read_parquet(
        '{behavior_path}'
    )

    GROUP BY
        user,
        day

    HAVING COUNT(*) > 1
)
""").fetchone()[0]


pre_cutoff_rows = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{behavior_path}'
)

WHERE day < DATE '{CUTOFF}'
""").fetchone()[0]


weekend_rows = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{behavior_path}'
)

WHERE is_weekend <> 0
""").fetchone()[0]


no_logon_rows = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{behavior_path}'
)

WHERE logon_count <= 0
""").fetchone()[0]


invalid_active_channels = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{behavior_path}'
)

WHERE active_channel_count < 0
   OR active_channel_count > 3
""").fetchone()[0]

con.close()


print("=" * 100)
print("BEHAVIOR FEATURE CORE VALIDATION")
print("=" * 100)

print(
    summary.to_string(
        index=False
    )
)

print("\nDuplicate user-days :", duplicates)
print("Pre-cutoff rows     :", pre_cutoff_rows)
print("Weekend rows        :", weekend_rows)
print("Rows without logon  :", no_logon_rows)
print("Bad channel counts  :", invalid_active_channels)


assert duplicates == 0
assert pre_cutoff_rows == 0
assert weekend_rows == 0
assert no_logon_rows == 0
assert invalid_active_channels == 0


print(
    "\n✅ SUCCESS — BEHAVIOR FEATURE CORE VALIDATION PASSED."
)

BEHAVIOR FEATURE CORE VALIDATION
  rows  users  first_day   last_day  min_baseline_workdays  median_baseline_workdays  max_baseline_workdays
387162   3737 2011-01-03 2011-06-01                    249                     251.0                    260

Duplicate user-days : 0
Pre-cutoff rows     : 0
Weekend rows        : 0
Rows without logon  : 0
Bad channel counts  : 0

✅ SUCCESS — BEHAVIOR FEATURE CORE VALIDATION PASSED.


In [9]:
malicious_sql = (
    "'ACM2278', "
    "'CMP2946', "
    "'PLJ1771', "
    "'CDE1846', "
    "'MBG3183'"
)

con = duckdb.connect()

malicious_present = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{behavior_path}'
)

WHERE user IN ({malicious_sql})
""").fetchone()[0]


missing_baseline = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{behavior_path}'
)

WHERE baseline_workdays IS NULL
""").fetchone()[0]

con.close()


print("Malicious rows present :", malicious_present)
print("Missing baseline rows   :", missing_baseline)

assert malicious_present == 0
assert missing_baseline == 0

print(
    "\n✅ SUCCESS — Baseline linkage is clean."
)

Malicious rows present : 0
Missing baseline rows   : 0

✅ SUCCESS — Baseline linkage is clean.


In [11]:
# =========================================================
# LOGICAL FEATURE SANITY
# =========================================================

con = duckdb.connect()

bad_logic = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{behavior_path}'
)

WHERE

       email_active_today NOT IN (0,1)
    OR usb_active_today NOT IN (0,1)
    OR http_upload_active_today NOT IN (0,1)

    OR email_count_above_p95 NOT IN (0,1)
    OR usb_copy_above_p95 NOT IN (0,1)
    OR http_upload_above_p95 NOT IN (0,1)

    OR email_active_days_5wd < 0
    OR email_active_days_5wd > 5

    OR usb_active_days_5wd < 0
    OR usb_active_days_5wd > 5

    OR http_active_days_5wd < 0
    OR http_active_days_5wd > 5

    OR email_active_days_10wd < 0
    OR email_active_days_10wd > 10

    OR usb_active_days_10wd < 0
    OR usb_active_days_10wd > 10

    OR http_active_days_10wd < 0
    OR http_active_days_10wd > 10

    OR email_active_days_20wd < 0
    OR email_active_days_20wd > 20

    OR usb_active_days_20wd < 0
    OR usb_active_days_20wd > 20

    OR http_active_days_20wd < 0
    OR http_active_days_20wd > 20

    OR multi_channel_days_10wd < 0
    OR multi_channel_days_10wd > 10

    OR active_channel_count < 0
    OR active_channel_count > 3
""").fetchone()[0]

con.close()

print(
    "Rows with impossible feature values:",
    bad_logic
)

assert bad_logic == 0

print(
    "\n✅ SUCCESS — Behavioral feature logic is valid."
)

Rows with impossible feature values: 0

✅ SUCCESS — Behavioral feature logic is valid.


In [13]:
# =========================================================
# BEHAVIOR FEATURE SCHEMA INSPECTION
# =========================================================

con = duckdb.connect()

schema = con.execute(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{behavior_path}'
)
""").df()

con.close()

print("=" * 100)
print("BEHAVIOR FEATURE SCHEMA")
print("=" * 100)

print(
    schema[
        [
            "column_name",
            "column_type",
            "null"
        ]
    ].to_string(index=False)
)

print("\nTotal columns:", len(schema))

print(
    "\n✅ SUCCESS — Behavior feature schema inspected."
)

BEHAVIOR FEATURE SCHEMA
               column_name column_type null
                      user     VARCHAR  YES
                       day        DATE  YES
               day_of_week     INTEGER  YES
                is_weekend     INTEGER  YES
         email_event_count      BIGINT  YES
          email_send_count      DOUBLE  YES
          email_view_count      DOUBLE  YES
      external_email_count      DOUBLE  YES
      internal_email_count      DOUBLE  YES
    total_sent_email_bytes      DOUBLE  YES
      external_email_bytes      DOUBLE  YES
   emails_with_attachments      DOUBLE  YES
          attachment_count      DOUBLE  YES
unique_external_recipients      BIGINT  YES
   unique_external_domains      BIGINT  YES
          file_event_count      BIGINT  YES
           file_copy_count      DOUBLE  YES
           file_open_count      DOUBLE  YES
          file_write_count      DOUBLE  YES
         file_delete_count      DOUBLE  YES
            usb_copy_count      DOUBLE  YES
      fr

In [15]:
# =========================================================
# REQUIRED FEATURE COLUMNS CHECK
# =========================================================

required_columns = {

    # Keys
    "user",
    "day",

    # Original channels
    "external_email_count",
    "external_email_bytes",
    "usb_copy_count",
    "www_upload_count",
    "logon_count",

    # Baseline
    "baseline_workdays",
    "email_active_rate",
    "usb_active_rate",
    "http_upload_active_rate",

    "email_count_p95",
    "email_bytes_p95",
    "usb_copy_p95",
    "http_upload_p95",

    # Today
    "email_active_today",
    "usb_active_today",
    "http_upload_active_today",
    "active_channel_count",

    # P95 anomaly features
    "email_count_above_p95",
    "email_bytes_above_p95",
    "usb_copy_above_p95",
    "http_upload_above_p95",

    # Ratios
    "email_count_to_p95_ratio",
    "email_bytes_to_p95_ratio",
    "usb_copy_to_p95_ratio",
    "http_upload_to_p95_ratio",

    # Rolling workday features
    "email_active_days_5wd",
    "email_active_days_10wd",
    "email_active_days_20wd",

    "usb_active_days_5wd",
    "usb_active_days_10wd",
    "usb_active_days_20wd",

    "http_active_days_5wd",
    "http_active_days_10wd",
    "http_active_days_20wd",

    "external_email_count_5wd",
    "external_email_count_10wd",

    "usb_copy_count_5wd",
    "usb_copy_count_10wd",

    "http_upload_count_5wd",
    "http_upload_count_10wd",

    "multi_channel_days_10wd"
}


con = duckdb.connect()

actual_columns = set(
    con.execute(f"""
        DESCRIBE
        SELECT *
        FROM read_parquet(
            '{behavior_path}'
        )
    """).df()["column_name"]
)

con.close()


missing_columns = (
    required_columns
    - actual_columns
)

unexpected_old_names = {
    col
    for col in actual_columns
    if col.endswith("_5w")
    or col.endswith("_10w")
    or col.endswith("_20w")
}


print("=" * 100)
print("REQUIRED FEATURE COLUMN CHECK")
print("=" * 100)

print("Required columns :", len(required_columns))
print("Actual columns   :", len(actual_columns))

print("\nMissing required columns:")
print(
    sorted(missing_columns)
    if missing_columns
    else "None"
)

print("\nOld '_w' names still present:")
print(
    sorted(unexpected_old_names)
    if unexpected_old_names
    else "None"
)


assert len(missing_columns) == 0
assert len(unexpected_old_names) == 0

print(
    "\n✅ SUCCESS — Required feature columns are present and naming is clean."
)

REQUIRED FEATURE COLUMN CHECK
Required columns : 43
Actual columns   : 85

Missing required columns:
None

Old '_w' names still present:
None

✅ SUCCESS — Required feature columns are present and naming is clean.


In [17]:
# =========================================================
# NULL AUDIT
# =========================================================

con = duckdb.connect()

critical_nulls = con.execute(f"""
SELECT

    SUM(
        CASE WHEN user IS NULL
        THEN 1 ELSE 0 END
    ) AS null_user,

    SUM(
        CASE WHEN day IS NULL
        THEN 1 ELSE 0 END
    ) AS null_day,

    SUM(
        CASE WHEN baseline_workdays IS NULL
        THEN 1 ELSE 0 END
    ) AS null_baseline_workdays,

    SUM(
        CASE WHEN external_email_count IS NULL
        THEN 1 ELSE 0 END
    ) AS null_external_email_count,

    SUM(
        CASE WHEN usb_copy_count IS NULL
        THEN 1 ELSE 0 END
    ) AS null_usb_copy_count,

    SUM(
        CASE WHEN www_upload_count IS NULL
        THEN 1 ELSE 0 END
    ) AS null_www_upload_count,

    SUM(
        CASE WHEN active_channel_count IS NULL
        THEN 1 ELSE 0 END
    ) AS null_active_channel_count

FROM read_parquet(
    '{behavior_path}'
)
""").df()


baseline_null_summary = con.execute(f"""
SELECT

    SUM(
        CASE WHEN email_count_p95 IS NULL
        THEN 1 ELSE 0 END
    ) AS null_email_p95,

    SUM(
        CASE WHEN usb_copy_p95 IS NULL
        THEN 1 ELSE 0 END
    ) AS null_usb_p95,

    SUM(
        CASE WHEN http_upload_p95 IS NULL
        THEN 1 ELSE 0 END
    ) AS null_http_p95

FROM read_parquet(
    '{behavior_path}'
)
""").df()

con.close()


print("=" * 100)
print("CRITICAL NULL CHECK")
print("=" * 100)

print(
    critical_nulls.to_string(
        index=False
    )
)

print("\n" + "=" * 100)
print("EXPECTED BASELINE NULLS")
print("=" * 100)

print(
    baseline_null_summary.to_string(
        index=False
    )
)


assert (
    critical_nulls.iloc[0] == 0
).all()

print(
    "\n✅ SUCCESS — No critical NULL values found."
)

CRITICAL NULL CHECK
 null_user  null_day  null_baseline_workdays  null_external_email_count  null_usb_copy_count  null_www_upload_count  null_active_channel_count
       0.0       0.0                     0.0                        0.0                  0.0                    0.0                        0.0

EXPECTED BASELINE NULLS
 null_email_p95  null_usb_p95  null_http_p95
            0.0      312211.0       328257.0

✅ SUCCESS — No critical NULL values found.


In [19]:
# =========================================================
# EXPECTED NULL CONSISTENCY CHECK
# =========================================================

con = duckdb.connect()

null_consistency = con.execute(f"""
SELECT

    SUM(
        CASE
            WHEN email_count_p95 IS NULL
             AND email_active_days > 0
            THEN 1 ELSE 0
        END
    ) AS bad_email_null,

    SUM(
        CASE
            WHEN usb_copy_p95 IS NULL
             AND usb_active_days > 0
            THEN 1 ELSE 0
        END
    ) AS bad_usb_null,

    SUM(
        CASE
            WHEN http_upload_p95 IS NULL
             AND http_upload_active_days > 0
            THEN 1 ELSE 0
        END
    ) AS bad_http_null

FROM read_parquet(
    '{behavior_path}'
)
""").df()

con.close()

print("=" * 100)
print("EXPECTED NULL CONSISTENCY")
print("=" * 100)

print(
    null_consistency.to_string(
        index=False
    )
)

assert (
    null_consistency.iloc[0] == 0
).all()

print(
    "\n✅ SUCCESS — All baseline NULLs are logically justified."
)

EXPECTED NULL CONSISTENCY
 bad_email_null  bad_usb_null  bad_http_null
            0.0           0.0            0.0

✅ SUCCESS — All baseline NULLs are logically justified.
